# Phase 2B — Colab verification and one OptiMind attempt
Live verification is PENDING until this notebook actually runs. Use a GPU runtime with sufficient full-weight memory. No automatic quantization or CPU offload occurs. Manual upload must include src, config, schemas, tests, requirements*.txt, docs and this notebook. Native generated Python is saved as `.py.txt` and never executed. Select one experiment by default; E1/E2 are native autonomous/template-guided and E3/E4 are IR autonomous/template-guided. Each has one attempt and no repair.

Use MAX_NEW_TOKENS=8192 consistently for the comparison pilot. Select one experiment. Each launch consumes RUN_INFERENCE and prints only its new attempt. Do not rerun staging to switch experiments; change SELECTED_EXPERIMENTS and opt in in a separate cell. Active-process detection is a notebook safeguard, not a supervisor or orphan recovery mechanism. Never remove a lock or terminate a process without checking its ownership.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys, subprocess, shutil, uuid, json, os
DRIVE_PROJECT = Path('/content/drive/MyDrive/workload-placement-research')
RUN_INFERENCE = False  # Explicitly opt in only after reviewing preparation results.
PINNED_AI_SOURCE = DRIVE_PROJECT / 'pinned_ai.yaml'  # Copy the FAILED E1 run's original pinned file here.
SELECTED_EXPERIMENTS = ['E1']  # Choose ONE: E1, E2, E3 or E4.
MAX_NEW_TOKENS = 8192  # Common comparison budget; applied to a new YAML, never the original pin.
HF_TOKEN_SECRET_NAME = 'HF_TOKEN'  # Put the token in Colab Secrets, never notebook source.
BENCHMARK = 'tests/fixtures/config/scenarios/single.yaml'
AI_CONFIG = 'tests/fixtures/config/ai/optimind_v2.yaml'
RUNTIME_PROJECT = Path('/content') / ('optimind-research-' + uuid.uuid4().hex[:8])
assert DRIVE_PROJECT.is_dir(), 'Manually upload the project first.'
shutil.copytree(DRIVE_PROJECT, RUNTIME_PROJECT, ignore=shutil.ignore_patterns('results', '.venv', '.git', '__pycache__', '.pytest_cache', '.model_cache'))
os.chdir(RUNTIME_PROJECT)
sys.path.insert(0, str(RUNTIME_PROJECT))
print('Python:', sys.executable, sys.version)
print('Staged project:', RUNTIME_PROJECT)
assert (RUNTIME_PROJECT / 'tests/suite_manifest.json').is_file(), 'Incomplete upload: missing tests/suite_manifest.json'
RESULTS = RUNTIME_PROJECT / 'results'
CACHE = RUNTIME_PROJECT / '.model_cache'
# Baseline packages only. No inference packages/weights yet.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
# This runs the complete suite, three Phase 2A demos, a restricted-license
# check and actual Drive copy/hash verification. Failure stops inference.
from src.runtime_verification import sync_verified
try:
    subprocess.run([sys.executable, '-m', 'src.runtime_verification', '--output', str(RESULTS / 'runtime'),
                    '--drive-output', str(DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name / 'runtime')], check=True)
finally:
    if RESULTS.exists():
        sync_verified(RESULTS, DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name)
RUNTIME_EVIDENCE = next((RESULTS / 'runtime').glob('runtime-*'))
assert json.loads((RUNTIME_EVIDENCE / 'runtime.json').read_text())['ready_for_inference']


In [ ]:
# Optional Hugging Face authentication. Create a Colab Secret named HF_TOKEN
# and enable notebook access. Never paste the token here or print it.
from google.colab import userdata
if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get(HF_TOKEN_SECRET_NAME)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print('HF token unavailable: enable the named Colab Secret before large downloads to reduce anonymous rate limiting.')
print('HF authentication configured:', bool(os.environ.get('HF_TOKEN')))


In [ ]:
# Fetch only single checkpoint metadata, pin its SHA, and size all four prompts.
# This is before optional inference dependencies and model weights.
import shutil
import yaml
import re

assert PINNED_AI_SOURCE.is_file(), (
    "Supply the original E1 pinned_ai.yaml; do not resolve main again."
)

prior_settings = yaml.safe_load(PINNED_AI_SOURCE.read_text())
revision = prior_settings.get("model", {}).get("revision")
assert isinstance(revision, str) and re.fullmatch(r"[a-f0-9]{40}", revision), (
    "Original checkpoint must be pinned."
)

destination = DRIVE_PROJECT / "pinned_ai.yaml"

if destination.exists():
    assert destination.read_bytes() == PINNED_AI_SOURCE.read_bytes(), (
        "A different pinned configuration already exists; review before replacing."
    )
else:
    shutil.copy2(PINNED_AI_SOURCE, destination)

PINNED_AI_SOURCE = destination
# A changed budget is new experimental configuration, not a patch to old evidence.
from copy import deepcopy
from src.inference_config import InferenceAI
from src.persistence import hash_file
settings = deepcopy(prior_settings)
settings['inference']['max_new_tokens'] = MAX_NEW_TOKENS
InferenceAI.model_validate(settings)
PREPARATION = RESULTS / ('preparation-' + uuid.uuid4().hex[:12])
PREPARATION.mkdir(parents=True, exist_ok=False)
RUN_CONFIG = PREPARATION / 'ai_config.yaml'
RUN_CONFIG.write_text(yaml.safe_dump(settings, sort_keys=False))
(PREPARATION / 'configuration_change.json').write_text(json.dumps({
    'source_sha256': hash_file(PINNED_AI_SOURCE),
    'revision': revision,
    'max_new_tokens_before': prior_settings['inference']['max_new_tokens'],
    'max_new_tokens_after': MAX_NEW_TOKENS,
}, indent=2))
print('Preparing output budget:', MAX_NEW_TOKENS, 'Revision:', revision)
try:
    subprocess.run([sys.executable, '-m', 'tests.support.prepare_inference', '--config', BENCHMARK,
                    '--ai-config', str(RUN_CONFIG), '--runtime-evidence', str(RUNTIME_EVIDENCE),
                    '--output', str(PREPARATION / 'preflight'), '--cache-dir', str(CACHE)], check=True)
finally:
    sync_verified(RESULTS, DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name)
PINNED_AI = PREPARATION / 'preflight' / 'pinned_ai.yaml'
assert PINNED_AI.exists()


In [ ]:
# Install only the explicitly selected candidate backend. AUTO requires an
# earlier genuine backend receipt, so use an explicit backend for bootstrap.
import yaml
backend = yaml.safe_load(PINNED_AI.read_text())['inference']['backend']
requirements = {'TRANSFORMERS': 'requirements-inference-transformers.txt',
                'SGLANG': 'requirements-inference-sglang.txt',
                'VLLM': 'requirements-inference-vllm.txt'}
assert backend in requirements, 'Choose an explicit backend for the first live attempt.'
try:
    # Capture the original incompatibility before changing packages.
    if backend == 'TRANSFORMERS':
        subprocess.run([sys.executable, '-m', 'src.text_environment', '--output', str(RESULTS / 'text_environment_before.json')])
    with (RESULTS / 'inference_install.log').open('w') as log:
        if backend == 'TRANSFORMERS':
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], stdout=log, stderr=subprocess.STDOUT, check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', requirements[backend], '-c', 'requirements.txt'],
                       stdout=log, stderr=subprocess.STDOUT, check=True)
    if backend == 'TRANSFORMERS':
        # Fresh subprocess forces all three Transformers lazy imports, without weights.
        subprocess.run([sys.executable, '-m', 'src.text_environment', '--output', str(RESULTS / 'text_environment_after.json')], check=True)
    with (RESULTS / 'pip_freeze.txt').open('w') as log:
        subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=log, check=True)
    # Recheck baseline behavior in a fresh process after package installation.
    subprocess.run([sys.executable, '-m', 'src.runtime_verification', '--output', str(RESULTS / 'post_install_runtime'),
                    '--drive-output', str(DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name / 'post_install_runtime')], check=True)
    RUNTIME_EVIDENCE = next((RESULTS / 'post_install_runtime').glob('runtime-*'))
finally:
    sync_verified(RESULTS, DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name)


In [ ]:
if not RUN_INFERENCE:
    print('Preparation only: inference disabled. Set RUN_INFERENCE=True in a new cell when ready; do not rerun staging.')
else:
    # Consume the opt-in immediately; rerunning this cell cannot silently repeat inference.
    RUN_INFERENCE = False
    import yaml
    assert len(SELECTED_EXPERIMENTS) == 1 and SELECTED_EXPERIMENTS[0] in ('E1', 'E2', 'E3', 'E4'), 'Select exactly one experiment.'
    effective = yaml.safe_load(PINNED_AI.read_text())
    assert effective['inference']['max_new_tokens'] == MAX_NEW_TOKENS, 'Budget changed: rerun metadata preparation first.'
    # Inspect actual process arguments, including orphaned workers from prior attempts.
    active = []
    for process in Path('/proc').iterdir():
        if process.name.isdigit():
            try:
                args = (process / 'cmdline').read_bytes().split(b'\0')
                if any(module in args for module in (b'src.inference_worker', b'tests.support.optimind_experiment')):
                    active.append(process.name)
            except (FileNotFoundError, ProcessLookupError):
                pass
    assert not active, f'Existing inference process(es): {active}. Inspect before starting another attempt.'
    lock = RUNTIME_PROJECT / '.notebook_inference.lock'
    # An exclusive lock prevents overlapping launches from this staged project.
    with lock.open('x') as handle:
        handle.write(str(os.getpid()))
    statuses = {}
    run_output = RESULTS / ('execution-' + uuid.uuid4().hex[:12])
    print('Experiment:', SELECTED_EXPERIMENTS[0], 'Output budget:', effective['inference']['max_new_tokens'])
    print('Checkpoint:', effective['model']['revision'], 'Results:', run_output)
    try:
        for experiment in SELECTED_EXPERIMENTS:
            completed = subprocess.run([sys.executable, '-m', 'tests.support.optimind_experiment', '--config', BENCHMARK,
                                        '--ai-config', str(PINNED_AI), '--experiment', experiment,
                                        '--runtime-evidence', str(RUNTIME_EVIDENCE), '--cache-dir', str(CACHE),
                                        '--output', str(run_output)])
            statuses[experiment] = completed.returncode
    finally:
        lock.unlink()
        sync_verified(RESULTS, DRIVE_PROJECT / 'results' / RUNTIME_PROJECT.name)
    print(statuses)
    for path in sorted(run_output.glob('*/attempt.json')):
        attempt = json.loads(path.read_text())
        print('Attempt:', path.parent)
        for key in ('experiment_id', 'status', 'genuine_inference', 'finish_reason', 'output_tokens', 'failure', 'evaluation'):
            print(key, ':', attempt.get(key))
